# PhoBERT v2: cleaned data, with and without ViHSD

Setup: **GPU T4 x2**, **Internet On**, secret `HF_TOKEN`, input dataset = **`vfyr-clean-splits.zip`**:

- `clean/` – leakage-free splits; likely label errors (confident learning) pruned from train only
- `clean_vihsd/` – `clean/` + 22.5k comments from the public UIT-ViHSD train/dev added to train
- `vihsd_test/` – ViHSD test set, an out-of-domain benchmark (never trained on)

Val/test are identical in `clean/` and `clean_vihsd/`, so runs C and D are directly comparable.
Both use **no class weights** (v1 over-predicted *offensive*) and up to 6 epochs with early stopping.

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji
import glob, os, shutil, json
for variant, splits in [("clean", ["train", "val", "test"]), ("clean_vihsd", ["train", "val", "test"]), ("vihsd_test", ["test"])]:
    os.makedirs(f"data/{variant}", exist_ok=True)
    for split in splits:
        src = glob.glob(f"/kaggle/input/**/{variant}/{split}.csv", recursive=True)
        assert len(src) == 1, (variant, split, src)
        shutil.copy(src[0], f"data/{variant}/{split}.csv")
print({v: os.listdir(f"data/{v}") for v in ["clean", "clean_vihsd", "vihsd_test"]})

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
# Run C: our cleaned data only
!python scripts/train_phobert.py --data-dir data/clean --no-class-weights --epochs 6 --output-dir models/C --tag _clean_nocw --no-push

In [ ]:
# Run D: our cleaned data + ViHSD train/dev
!python scripts/train_phobert.py --data-dir data/clean_vihsd --no-class-weights --epochs 6 --output-dir models/D --tag _clean_vihsd_nocw --no-push

In [ ]:
# Out-of-domain benchmark: ViHSD test
!python scripts/evaluate_model.py --model models/C/best --data-dir data/vihsd_test --name phobert_C --tag _vihsd_test
!python scripts/evaluate_model.py --model models/D/best --data-dir data/vihsd_test --name phobert_D --tag _vihsd_test

In [ ]:
c = json.load(open("reports/phobert_results_clean_nocw.json"))
d = json.load(open("reports/phobert_results_clean_vihsd_nocw.json"))
c["vihsd_test"] = json.load(open("reports/phobert_C_eval_vihsd_test.json"))
d["vihsd_test"] = json.load(open("reports/phobert_D_eval_vihsd_test.json"))
for name, r in [("C clean", c), ("D clean+ViHSD", d)]:
    print(f"{name}: val F1={r['val_f1_macro']} | test F1={r['test']['f1_macro']} "
          f"off={r['test']['f1_per_class']['offensive']} | ViHSD test F1={r['vihsd_test']['f1_macro']}")
# Pick on validation only. D is preferred unless clearly worse (it generalizes to other sources).
best = "models/D/best" if d["val_f1_macro"] >= c["val_f1_macro"] - 0.005 else "models/C/best"
print("Pushing", best)
from huggingface_hub import HfApi
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
HfApi().upload_folder(folder_path=best, repo_id=repo_id, commit_message=f"v2: trained on {best}")
print(f"https://huggingface.co/{repo_id}")

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps({"C_clean_nocw": c, "D_clean_vihsd_nocw": d}, ensure_ascii=False))
